# 🧠 UEBA Lab — Notebook 03: LLM Reasoning on PCAI

**Input:** `flagged_users.csv` (output of Notebook 02)
**Goal:** Connect to the HPE Private Cloud AI LLM endpoint, build structured
chain-of-thought prompts from ML evidence, and generate analyst-ready SOC alerts.

## 📋 Notebook Sections
| Section | Topic |
|---------|-------|
| 3.0 | Setup & Load Flagged Users |
| 3.1 | PCAI LLM Client Setup |
| 3.2 | Prompt Engineering |
| 3.3 | Batch Alert Generation |
| 3.4 | Alert Quality Review |
| 3.5 | Save outputs |

> **⚠️ Before running:** Complete Notebook 02 to generate `flagged_users.csv`.
> Set your PCAI endpoint and API key in Section 3.1.

---
## 3.0 Setup & Load Flagged Users

In [ ]:
import os
import ast
import json
import time
import warnings
import textwrap
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm import tqdm
from datetime import datetime
from typing import Optional

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 200)

print('✅ Imports loaded.')

In [ ]:
# ── Load flagged users ────────────────────────────────────────
FLAGGED_PATH = 'flagged_users.csv'

if not os.path.exists(FLAGGED_PATH):
    raise FileNotFoundError(
        f'❌ {FLAGGED_PATH} not found.\n'
        f'   Please run Notebook 02 first.'
    )

flagged = pd.read_csv(FLAGGED_PATH, parse_dates=['date_only'])
flagged = flagged.sort_values('risk_score', ascending=False).reset_index(drop=True)

print(f'✅ Flagged users loaded: {len(flagged):,} records')
print(f'   Alert level breakdown:')
print(flagged['alert_level'].value_counts().to_string())
print(f'\n   Risk score range : [{flagged["risk_score"].min():.4f}, '
      f'{flagged["risk_score"].max():.4f}]')
flagged[[
    'user_id', 'date_only', 'risk_score',
    'alert_level', 'iso_score', 'ae_score', 'lstm_score'
]].head(5)

In [ ]:
# ── Load risk_scores for baseline delta context ───────────────
RISK_SCORES_PATH = 'risk_scores.csv'

if os.path.exists(RISK_SCORES_PATH):
    risk_scores = pd.read_csv(RISK_SCORES_PATH, parse_dates=['date_only'])
    print(f'✅ risk_scores.csv loaded: {risk_scores.shape}')
else:
    risk_scores = None
    print('⚠️  risk_scores.csv not found — baseline delta context will be skipped.')

---
## 3.1 PCAI LLM Client Setup

Configure your HPE Private Cloud AI endpoint below.
The client uses the OpenAI-compatible `/v1/chat/completions` API.

In [ ]:
# ── PCAI Configuration — edit these values ────────────────────
PCAI_ENDPOINT  = 'https://<your-pcai-host>/v1/chat/completions'
PCAI_API_KEY   = '<your-pcai-bearer-token>'
PCAI_MODEL     = 'llama3-70b'

# Generation parameters
LLM_TEMPERATURE  = 0.1    # Low temperature for consistent structured output
LLM_MAX_TOKENS   = 900
LLM_TOP_P        = 0.9
LLM_TIMEOUT_SEC  = 60

# Batch processing
BATCH_DELAY_SEC  = 0.5    # Polite delay between API calls
MAX_RETRIES      = 3
RETRY_DELAY_SEC  = 5

print('✅ PCAI configuration set.')
print(f'   Endpoint : {PCAI_ENDPOINT}')
print(f'   Model    : {PCAI_MODEL}')
print(f'   Temp     : {LLM_TEMPERATURE}')
print(f'   Max tok  : {LLM_MAX_TOKENS}')

In [ ]:
# ── LLM client function ───────────────────────────────────────
def call_llm(
    system_prompt: str,
    user_prompt: str,
    temperature: float = LLM_TEMPERATURE,
    max_tokens: int = LLM_MAX_TOKENS,
    retries: int = MAX_RETRIES,
) -> Optional[str]:
    """
    Call the PCAI LLM endpoint with a system + user prompt.

    Returns:
        The assistant message content string, or None on failure.
    """
    headers = {
        'Authorization': f'Bearer {PCAI_API_KEY}',
        'Content-Type' : 'application/json',
    }
    payload = {
        'model'      : PCAI_MODEL,
        'temperature': temperature,
        'max_tokens' : max_tokens,
        'top_p'      : LLM_TOP_P,
        'messages'   : [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_prompt},
        ],
    }

    for attempt in range(1, retries + 1):
        try:
            response = requests.post(
                PCAI_ENDPOINT,
                headers=headers,
                json=payload,
                timeout=LLM_TIMEOUT_SEC,
            )
            response.raise_for_status()
            data = response.json()
            return data['choices'][0]['message']['content']

        except requests.exceptions.Timeout:
            print(f'   ⚠️  Attempt {attempt}/{retries}: Timeout after '
                  f'{LLM_TIMEOUT_SEC}s')
        except requests.exceptions.HTTPError as e:
            print(f'   ⚠️  Attempt {attempt}/{retries}: HTTP {e.response.status_code}')
            if e.response.status_code in [400, 401, 403]:
                print('   ❌ Non-retryable error. Check endpoint/API key.')
                return None
        except Exception as e:
            print(f'   ⚠️  Attempt {attempt}/{retries}: {type(e).__name__}: {e}')

        if attempt < retries:
            time.sleep(RETRY_DELAY_SEC)

    print(f'   ❌ All {retries} attempts failed.')
    return None


print('✅ LLM client function defined.')

In [ ]:
# ── Connection test ───────────────────────────────────────────
print('🔌 Testing PCAI LLM connection...\n')

test_response = call_llm(
    system_prompt='You are a helpful assistant. Reply in one sentence.',
    user_prompt='Confirm you are online and ready for UEBA alert generation.',
    max_tokens=60,
)

if test_response:
    print(f'✅ Connection successful!')
    print(f'   Model response: {test_response.strip()}')
else:
    print('❌ Connection failed.')
    print('   Check PCAI_ENDPOINT and PCAI_API_KEY in cell 3.1.')

---
## 3.2 Prompt Engineering

Good prompt design is the difference between a generic LLM response and
an analyst-grade SOC alert. We use three components:

1. **System prompt** — establishes the LLM's role as a senior threat analyst
2. **Context builder** — transforms ML features + SHAP values into a structured evidence block
3. **Chain-of-thought user prompt** — instructs the LLM to reason step-by-step
   before producing a structured JSON alert

### Output Schema
```json
{
  "threat_type"       : "<Data Exfiltration | Sabotage | Privilege Abuse | Reconnaissance | Benign>",
  "confidence"        : "<High | Medium | Low>",
  "summary"           : "<2-3 sentence plain-English summary>",
  "key_evidence"      : ["<evidence item 1>", "<evidence item 2>", ...],
  "recommended_action": "<immediate action for SOC analyst>",
  "false_positive_risk": "<High | Medium | Low>",
  "false_positive_reason": "<reason if false_positive_risk is High or Medium>",
  "kill_chain_stage"  : "<Reconnaissance | Weaponization | Delivery | Exploitation | Installation | C2 | Exfiltration | Unknown>"
}
```

In [ ]:
# ── System prompt ─────────────────────────────────────────────
SYSTEM_PROMPT = textwrap.dedent("""
    You are a senior cybersecurity threat analyst specialising in
    User and Entity Behavior Analytics (UEBA) and insider threat detection.

    Your task is to analyse ML-generated anomaly evidence for a flagged
    employee and produce a structured SOC alert in valid JSON format.

    Guidelines:
    - Reason carefully from the evidence provided. Do not invent facts.
    - Consider benign explanations (travel, project deadlines, role changes)
      before concluding malicious intent.
    - Classify the most likely threat type from the evidence.
    - Assess false positive risk honestly.
    - Keep the summary concise and actionable for a SOC analyst.
    - Your entire response must be a single valid JSON object.
      Do not include markdown fences, commentary, or any text outside the JSON.

    Required JSON fields:
    {
      "threat_type"            : string,
      "confidence"             : string,
      "summary"                : string,
      "key_evidence"           : list of strings,
      "recommended_action"     : string,
      "false_positive_risk"    : string,
      "false_positive_reason"  : string,
      "kill_chain_stage"       : string
    }
""").strip()

print('✅ System prompt defined.')
print(f'   Length: {len(SYSTEM_PROMPT)} characters')

In [ ]:
# ── Context builder ───────────────────────────────────────────
def build_context(
    row: pd.Series,
    risk_scores_df: Optional[pd.DataFrame] = None,
) -> str:
    """
    Build a structured evidence block from a flagged user row.
    Includes identity, ML scores, SHAP top-5, and optional
    30-day baseline delta context.

    Args:
        row            : One row from flagged_users.csv
        risk_scores_df : Full risk_scores.csv for baseline context

    Returns:
        Formatted evidence string for the user prompt.
    """
    lines = []

    # ── Identity block ────────────────────────────────────────
    lines.append('=== EMPLOYEE PROFILE ===')
    lines.append(f'User ID     : {row["user_id"]}')
    if 'role' in row and pd.notna(row.get('role')):
        lines.append(f'Role        : {row["role"]}')
    if 'department' in row and pd.notna(row.get('department')):
        lines.append(f'Department  : {row["department"]}')
    lines.append(f'Peak Risk Day: {str(row["date_only"])[:10]}')

    # ── ML Score block ────────────────────────────────────────
    lines.append('')
    lines.append('=== ML ANOMALY SCORES (0=normal, 1=anomalous) ===')
    lines.append(f'Ensemble Risk Score : {row["risk_score"]:.4f}  '
                 f'[{row["alert_level"].upper()}]')
    lines.append(f'Isolation Forest    : {row["iso_score"]:.4f}')
    lines.append(f'Autoencoder         : {row["ae_score"]:.4f}')
    lines.append(f'LSTM Sequence       : {row["lstm_score"]:.4f}')

    # ── Alert history block ───────────────────────────────────
    lines.append('')
    lines.append('=== ALERT HISTORY (last 30 days) ===')
    for level in ['critical_days', 'elevated_days', 'watch_days']:
        if level in row and pd.notna(row.get(level)):
            label = level.replace('_days', '').upper()
            lines.append(f'{label:<12}: {int(row[level])} day(s)')

    # ── SHAP top-5 block ──────────────────────────────────────
    lines.append('')
    lines.append('=== TOP ANOMALY DRIVERS (SHAP) ===')
    if 'shap_top5' in row and pd.notna(row.get('shap_top5')):
        try:
            shap_list = ast.literal_eval(str(row['shap_top5']))
            for i, item in enumerate(shap_list, 1):
                direction = (
                    'raises risk'
                    if float(item.get('shap_value', 0)) > 0
                    else 'lowers risk'
                )
                lines.append(
                    f'{i}. {item["feature"]:<35} '
                    f'value={item["feat_value"]:>8.4f}  '
                    f'shap={item["shap_value"]:>8.5f}  '
                    f'[{direction}]'
                )
        except Exception:
            lines.append(f'   {str(row["shap_top5"])[:300]}')
    else:
        lines.append('   (SHAP data not available)')

    # ── Behavioral feature snapshot ───────────────────────────
    lines.append('')
    lines.append('=== BEHAVIORAL FEATURE SNAPSHOT (peak risk day) ===')
    key_features = [
        'files_accessed', 'sensitive_file_ratio', 'delete_ratio',
        'after_hours_ratio', 'usb_event_count', 'usb_bytes_gb',
        'email_count', 'external_email_ratio', 'emails_with_attachments',
        'job_site_visits', 'cloud_storage_visits', 'webmail_visits',
        'login_count', 'unique_pcs', 'session_span_hours',
    ]
    for feat in key_features:
        if feat in row and pd.notna(row.get(feat)):
            lines.append(f'  {feat:<35}: {float(row[feat]):>10.4f}')

    # ── Baseline delta block ──────────────────────────────────
    if risk_scores_df is not None:
        user_history = risk_scores_df[
            risk_scores_df['user_id'] == row['user_id']
        ].sort_values('date_only')

        if len(user_history) >= 7:
            recent_7d  = user_history.tail(7)['risk_score'].mean()
            baseline_30d = user_history.tail(30)['risk_score'].mean()
            delta = recent_7d - baseline_30d
            trend = 'INCREASING ↑' if delta > 0.05 \
                    else 'DECREASING ↓' if delta < -0.05 \
                    else 'STABLE →'
            lines.append('')
            lines.append('=== RISK TREND ===')
            lines.append(
                f'7-day avg risk   : {recent_7d:.4f}'
            )
            lines.append(
                f'30-day avg risk  : {baseline_30d:.4f}'
            )
            lines.append(
                f'Delta            : {delta:+.4f}  [{trend}]'
            )

    return '\n'.join(lines)


print('✅ Context builder defined.')

# Preview context for top-risk user
print('\n📋 Sample context for top-risk user:')
print('-' * 60)
sample_context = build_context(flagged.iloc[0], risk_scores)
print(sample_context)
print('-' * 60)

In [ ]:
# ── Chain-of-thought user prompt template ─────────────────────
def build_user_prompt(context: str) -> str:
    """
    Wrap the evidence context in a chain-of-thought prompt
    that instructs the LLM to reason before producing JSON.
    """
    return textwrap.dedent(f"""
        Below is the ML-generated anomaly evidence for a flagged employee.
        Analyse this evidence carefully using the following reasoning steps:

        STEP 1 — EVIDENCE REVIEW
        Identify which behavioral signals are most significant and why.

        STEP 2 — BENIGN HYPOTHESIS
        Consider whether legitimate business activities (e.g., project
        deadlines, role changes, travel, IT tasks) could explain the anomalies.

        STEP 3 — THREAT HYPOTHESIS
        If the evidence suggests malicious intent, identify the most likely
        threat type and kill chain stage.

        STEP 4 — CONFIDENCE ASSESSMENT
        Weigh both hypotheses and assign confidence and false positive risk.

        STEP 5 — JSON OUTPUT
        Produce ONLY the final JSON object. No preamble, no markdown fences.

        --- EVIDENCE ---
        {context}
        --- END EVIDENCE ---

        Respond with the JSON alert object now:
    """).strip()


print('✅ Chain-of-thought prompt template defined.')
print(f'\nSample prompt length for top-risk user: '
      f'{len(build_user_prompt(sample_context))} characters')

In [ ]:
# ── JSON response parser ──────────────────────────────────────
REQUIRED_FIELDS = [
    'threat_type',
    'confidence',
    'summary',
    'key_evidence',
    'recommended_action',
    'false_positive_risk',
    'false_positive_reason',
    'kill_chain_stage',
]

VALID_THREAT_TYPES = [
    'Data Exfiltration',
    'Sabotage',
    'Privilege Abuse',
    'Reconnaissance',
    'Benign',
]

VALID_CONFIDENCE = ['High', 'Medium', 'Low']
VALID_FP_RISK    = ['High', 'Medium', 'Low']
VALID_KILL_CHAIN = [
    'Reconnaissance', 'Weaponization', 'Delivery',
    'Exploitation', 'Installation', 'C2',
    'Exfiltration', 'Unknown',
]


def parse_llm_response(
    raw: str,
    user_id: str,
) -> dict:
    """
    Parse and validate the LLM JSON response.

    - Strips markdown fences if present
    - Validates required fields
    - Normalises enum fields to title case
    - Returns a dict with a 'parse_status' key:
        'ok'      — fully valid
        'partial' — parsed but some fields missing/invalid
        'failed'  — could not parse JSON
    """
    if raw is None:
        return {
            'user_id': user_id,
            'parse_status': 'failed',
            'parse_error': 'LLM returned None',
        }

    # Strip markdown fences
    cleaned = raw.strip()
    if cleaned.startswith('```'):
        lines = cleaned.split('\n')
        cleaned = '\n'.join(
            l for l in lines
            if not l.strip().startswith('```')
        ).strip()

    # Extract JSON object
    start = cleaned.find('{')
    end   = cleaned.rfind('}') + 1
    if start == -1 or end == 0:
        return {
            'user_id': user_id,
            'parse_status': 'failed',
            'parse_error': 'No JSON object found in response',
            'raw_response': cleaned[:500],
        }

    try:
        alert = json.loads(cleaned[start:end])
    except json.JSONDecodeError as e:
        return {
            'user_id': user_id,
            'parse_status': 'failed',
            'parse_error': str(e),
            'raw_response': cleaned[:500],
        }

    alert['user_id'] = user_id
    issues = []

    # Validate required fields
    for field in REQUIRED_FIELDS:
        if field not in alert:
            alert[field] = 'Unknown'
            issues.append(f'missing: {field}')

    # Normalise enum fields
    for field, valid_set in [
        ('confidence',          VALID_CONFIDENCE),
        ('false_positive_risk', VALID_FP_RISK),
    ]:
        val = str(alert.get(field, '')).strip().title()
        alert[field] = val if val in valid_set else 'Unknown'

    # Ensure key_evidence is a list
    if not isinstance(alert.get('key_evidence'), list):
        alert['key_evidence'] = [str(alert.get('key_evidence', ''))]

    alert['parse_status'] = 'partial' if issues else 'ok'
    if issues:
        alert['parse_issues'] = issues

    return alert


print('✅ JSON response parser defined.')
print(f'   Required fields : {REQUIRED_FIELDS}')

In [ ]:
# ── Single-user alert test ────────────────────────────────────
print('🧪 Running single-user alert test on top-risk user...\n')

test_row     = flagged.iloc[0]
test_context = build_context(test_row, risk_scores)
test_prompt  = build_user_prompt(test_context)

print(f'   User ID     : {test_row["user_id"]}')
print(f'   Risk score  : {test_row["risk_score"]:.4f}')
print(f'   Alert level : {test_row["alert_level"].upper()}')
print(f'   Prompt len  : {len(test_prompt)} chars')
print('\n   Calling PCAI LLM...')

test_raw    = call_llm(SYSTEM_PROMPT, test_prompt)
test_alert  = parse_llm_response(test_raw, test_row['user_id'])

print(f'\n✅ Test alert generated. Parse status: '
      f'{test_alert["parse_status"].upper()}')
print('\n📋 Alert output:')
print(json.dumps(test_alert, indent=2))

---
## 3.3 Batch Alert Generation

We run the LLM on all flagged users, prioritising **critical** then **elevated**
alert levels. Each call is rate-limited with a configurable delay.

In [ ]:
# ── Batch configuration ───────────────────────────────────────
# Set MAX_USERS to None to process all flagged users
# Set to an integer (e.g. 20) for a quick workshop demo run
MAX_USERS = 20

# Process critical first, then elevated
priority_order = {'critical': 0, 'elevated': 1, 'watch': 2}
flagged_sorted = flagged.copy()
flagged_sorted['priority'] = flagged_sorted['alert_level'].map(
    priority_order
).fillna(3)
flagged_sorted = flagged_sorted.sort_values(
    ['priority', 'risk_score'],
    ascending=[True, False]
).reset_index(drop=True)

if MAX_USERS is not None:
    batch = flagged_sorted.head(MAX_USERS)
    print(f'⚠️  MAX_USERS={MAX_USERS}: processing top {MAX_USERS} users only.')
    print('   Set MAX_USERS = None to process all flagged users.')
else:
    batch = flagged_sorted

print(f'\n✅ Batch ready: {len(batch)} users to process')
print(f'   Alert level breakdown:')
print(batch['alert_level'].value_counts().to_string())
est_minutes = len(batch) * (LLM_TIMEOUT_SEC / 60 * 0.3 + BATCH_DELAY_SEC / 60)
print(f'\n   Estimated time: ~{est_minutes:.1f} minutes')

In [ ]:
# ── Batch alert generation loop ───────────────────────────────
alerts      = []
failed_ids  = []
start_time  = time.time()

print(f'🚀 Starting batch alert generation for {len(batch)} users...\n')

for idx, row in tqdm(
    batch.iterrows(),
    total=len(batch),
    desc='Generating alerts',
):
    user_id = row['user_id']

    # Build prompt
    context    = build_context(row, risk_scores)
    user_prompt = build_user_prompt(context)

    # Call LLM
    raw_response = call_llm(SYSTEM_PROMPT, user_prompt)

    # Parse response
    alert = parse_llm_response(raw_response, user_id)

    # Attach ML metadata
    alert['risk_score']   = float(row['risk_score'])
    alert['alert_level']  = str(row['alert_level'])
    alert['iso_score']    = float(row['iso_score'])
    alert['ae_score']     = float(row['ae_score'])
    alert['lstm_score']   = float(row['lstm_score'])
    alert['peak_risk_day'] = str(row['date_only'])[:10]
    alert['generated_at'] = datetime.utcnow().isoformat() + 'Z'

    if 'role' in row and pd.notna(row.get('role')):
        alert['role'] = str(row['role'])
    if 'department' in row and pd.notna(row.get('department')):
        alert['department'] = str(row['department'])

    alerts.append(alert)

    if alert['parse_status'] == 'failed':
        failed_ids.append(user_id)

    time.sleep(BATCH_DELAY_SEC)

elapsed = time.time() - start_time
ok_count      = sum(1 for a in alerts if a['parse_status'] == 'ok')
partial_count = sum(1 for a in alerts if a['parse_status'] == 'partial')
failed_count  = sum(1 for a in alerts if a['parse_status'] == 'failed')

print(f'\n✅ Batch complete in {elapsed:.1f}s '
      f'({elapsed/len(batch):.1f}s per user)')
print(f'   OK      : {ok_count}')
print(f'   Partial : {partial_count}')
print(f'   Failed  : {failed_count}')
if failed_ids:
    print(f'   Failed users: {failed_ids}')

In [ ]:
# ── Preview first 3 alerts ────────────────────────────────────
print('📋 Preview — first 3 generated alerts:\n')
for alert in alerts[:3]:
    print('=' * 65)
    print(f'  User         : {alert.get("user_id")}')
    print(f'  Risk Score   : {alert.get("risk_score", 0):.4f}  '
          f'[{alert.get("alert_level", "?").upper()}]')
    print(f'  Threat Type  : {alert.get("threat_type", "?")}')
    print(f'  Confidence   : {alert.get("confidence", "?")}')
    print(f'  Kill Chain   : {alert.get("kill_chain_stage", "?")}')
    print(f'  FP Risk      : {alert.get("false_positive_risk", "?")}')
    print(f'  Parse Status : {alert.get("parse_status", "?").upper()}')
    print(f'  Summary      :')
    summary = alert.get('summary', '')
    for line in textwrap.wrap(summary, width=60):
        print(f'    {line}')
    print(f'  Key Evidence :')
    for ev in alert.get('key_evidence', [])[:3]:
        for line in textwrap.wrap(str(ev), width=58):
            print(f'    • {line}')
    print(f'  Recommended  :')
    action = alert.get('recommended_action', '')
    for line in textwrap.wrap(action, width=60):
        print(f'    {line}')
print('=' * 65)

---
## 3.4 Alert Quality Review

We analyse the LLM alert batch across four dimensions:
1. **Threat type distribution** — what categories did the LLM identify?
2. **False positive filtering** — how many alerts did the LLM flag as likely benign?
3. **Confidence distribution** — how certain is the LLM across alert levels?
4. **ML score vs. LLM verdict** — do high ML scores correlate with high LLM confidence?

In [ ]:
# ── Build alerts DataFrame for analysis ──────────────────────
alerts_df = pd.DataFrame(alerts)

# Exclude failed parses from quality analysis
alerts_valid = alerts_df[alerts_df['parse_status'] != 'failed'].copy()

print(f'✅ Alerts DataFrame built: {alerts_df.shape}')
print(f'   Valid alerts for analysis: {len(alerts_valid)}')
print()
print('Threat type distribution:')
print(alerts_valid['threat_type'].value_counts().to_string())
print()
print('Confidence distribution:')
print(alerts_valid['confidence'].value_counts().to_string())
print()
print('False positive risk distribution:')
print(alerts_valid['false_positive_risk'].value_counts().to_string())

In [ ]:
# ── Quality review visualisations ────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Threat Type Distribution',
        'LLM Confidence Distribution',
        'False Positive Risk Distribution',
        'ML Risk Score vs LLM Confidence',
    ]
)

# 1. Threat type bar
threat_counts = alerts_valid['threat_type'].value_counts()
threat_colors = {
    'Data Exfiltration' : '#FF4444',
    'Sabotage'          : '#FF6B35',
    'Privilege Abuse'   : '#FFD700',
    'Reconnaissance'    : '#DA70D6',
    'Benign'            : '#00C853',
    'Unknown'           : '#888888',
}
fig.add_trace(
    go.Bar(
        x=threat_counts.index.tolist(),
        y=threat_counts.values.tolist(),
        marker_color=[
            threat_colors.get(t, '#888')
            for t in threat_counts.index
        ],
        name='Threat Type',
    ),
    row=1, col=1
)

# 2. Confidence donut
conf_counts = alerts_valid['confidence'].value_counts()
conf_colors = {'High': '#FF4444', 'Medium': '#FFD700',
               'Low': '#00C853', 'Unknown': '#888'}
fig.add_trace(
    go.Pie(
        labels=conf_counts.index.tolist(),
        values=conf_counts.values.tolist(),
        marker_colors=[
            conf_colors.get(c, '#888')
            for c in conf_counts.index
        ],
        hole=0.4,
        name='Confidence',
    ),
    row=1, col=2
)

# 3. False positive risk donut
fp_counts = alerts_valid['false_positive_risk'].value_counts()
fp_colors = {'High': '#00C853', 'Medium': '#FFD700',
             'Low': '#FF4444', 'Unknown': '#888'}
fig.add_trace(
    go.Pie(
        labels=fp_counts.index.tolist(),
        values=fp_counts.values.tolist(),
        marker_colors=[
            fp_colors.get(f, '#888')
            for f in fp_counts.index
        ],
        hole=0.4,
        name='FP Risk',
    ),
    row=2, col=1
)

# 4. ML risk score vs LLM confidence scatter
conf_numeric = {'High': 3, 'Medium': 2, 'Low': 1, 'Unknown': 0}
alerts_valid['conf_num'] = alerts_valid['confidence'].map(conf_numeric).fillna(0)
scatter_colors = [
    threat_colors.get(t, '#888')
    for t in alerts_valid['threat_type']
]
fig.add_trace(
    go.Scatter(
        x=alerts_valid['risk_score'],
        y=alerts_valid['conf_num'] + np.random.uniform(
            -0.1, 0.1, len(alerts_valid)
        ),
        mode='markers',
        marker=dict(
            size=8,
            color=scatter_colors,
            opacity=0.8,
        ),
        text=alerts_valid['user_id'],
        name='Users',
    ),
    row=2, col=2
)
fig.update_yaxes(
    tickvals=[1, 2, 3],
    ticktext=['Low', 'Medium', 'High'],
    row=2, col=2
)
fig.update_xaxes(title_text='ML Risk Score', row=2, col=2)
fig.update_yaxes(title_text='LLM Confidence', row=2, col=2)

fig.update_layout(
    height=700,
    plot_bgcolor='#0D1B2A',
    paper_bgcolor='#0D1B2A',
    font_color='white',
    showlegend=False,
    title_text='LLM Alert Quality Review',
    title_font_size=14,
)
fig.show()
print('✅ Alert quality review dashboard rendered.')

In [ ]:
# ── False positive filtering ──────────────────────────────────
# Identify alerts the LLM assessed as likely benign or high FP risk
fp_filter_mask = (
    (alerts_valid['threat_type'] == 'Benign') |
    (alerts_valid['false_positive_risk'] == 'High')
)

fp_filtered   = alerts_valid[fp_filter_mask].copy()
true_alerts   = alerts_valid[~fp_filter_mask].copy()

print('=' * 60)
print('  FALSE POSITIVE FILTERING SUMMARY')
print('=' * 60)
print(f'  Total valid alerts    : {len(alerts_valid):>5}')
print(f'  LLM-filtered as FP    : {len(fp_filtered):>5} '
      f'({len(fp_filtered)/len(alerts_valid)*100:.1f}%)')
print(f'  Actionable alerts     : {len(true_alerts):>5} '
      f'({len(true_alerts)/len(alerts_valid)*100:.1f}%)')
print('=' * 60)

if len(fp_filtered) > 0:
    print('\n  Sample filtered (likely benign) alerts:')
    for _, row in fp_filtered.head(3).iterrows():
        print(f'  • {row["user_id"]} | '
              f'risk={row["risk_score"]:.4f} | '
              f'FP reason: {str(row.get("false_positive_reason", ""))[:80]}')

In [ ]:
# ── Side-by-side: ML score vs LLM verdict table ───────────────
print('📊 ML Score vs LLM Verdict — Top 15 Actionable Alerts:\n')

comparison_cols = [
    'user_id', 'risk_score', 'alert_level',
    'threat_type', 'confidence',
    'false_positive_risk', 'kill_chain_stage',
]
comparison_cols = [
    c for c in comparison_cols if c in true_alerts.columns
]

comparison = (
    true_alerts[comparison_cols]
    .sort_values('risk_score', ascending=False)
    .head(15)
    .reset_index(drop=True)
)
comparison.index += 1
print(comparison.to_string())

print('\n💡 Key insight: LLM confidence does not always track ML risk score.')
print('   High ML score + Low LLM confidence = investigate context further.')
print('   High ML score + High LLM confidence = prioritise for SOC action.')

In [ ]:
# ── Kill chain stage distribution ────────────────────────────
kc_counts = (
    true_alerts['kill_chain_stage']
    .value_counts()
    .reset_index()
)
kc_counts.columns = ['stage', 'count']

# Ordered kill chain stages for x-axis
kc_order = [
    'Reconnaissance', 'Weaponization', 'Delivery',
    'Exploitation', 'Installation', 'C2',
    'Exfiltration', 'Unknown',
]
kc_counts['order'] = kc_counts['stage'].apply(
    lambda s: kc_order.index(s) if s in kc_order else 99
)
kc_counts = kc_counts.sort_values('order')

fig = px.bar(
    kc_counts,
    x='stage',
    y='count',
    title='Kill Chain Stage Distribution — Actionable Alerts',
    labels={'stage': 'Kill Chain Stage', 'count': 'Alert Count'},
    color='count',
    color_continuous_scale='Reds',
    height=380,
)
fig.update_layout(
    plot_bgcolor='#0D1B2A',
    paper_bgcolor='#0D1B2A',
    font_color='white',
    title_font_size=13,
    coloraxis_showscale=False,
    xaxis_tickangle=-20,
)
fig.show()
print('✅ Kill chain distribution rendered.')

---
## 3.5 Save Outputs

We save two artefacts for Notebook 04:
- `alerts.json` — all generated alerts (full detail, including parse metadata)
- `alerts_summary.csv` — flat summary table for dashboard and validation

In [ ]:
# ── Save alerts.json ──────────────────────────────────────────
ALERTS_JSON_PATH = 'alerts.json'

output = {
    'generated_at'     : datetime.utcnow().isoformat() + 'Z',
    'model'            : PCAI_MODEL,
    'total_alerts'     : len(alerts),
    'ok_count'         : ok_count,
    'partial_count'    : partial_count,
    'failed_count'     : failed_count,
    'fp_filtered_count': len(fp_filtered),
    'actionable_count' : len(true_alerts),
    'alerts'           : alerts,
}

with open(ALERTS_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, default=str)

print(f'✅ alerts.json saved.')
print(f'   Path  : {ALERTS_JSON_PATH}')
print(f'   Size  : {os.path.getsize(ALERTS_JSON_PATH)/1e3:.1f} KB')
print(f'   Total alerts in file: {len(alerts)}')

In [ ]:
# ── Save alerts_summary.csv ───────────────────────────────────
ALERTS_CSV_PATH = 'alerts_summary.csv'

summary_cols = [
    'user_id', 'peak_risk_day', 'risk_score', 'alert_level',
    'iso_score', 'ae_score', 'lstm_score',
    'threat_type', 'confidence', 'kill_chain_stage',
    'false_positive_risk', 'false_positive_reason',
    'summary', 'recommended_action', 'parse _status', 'generated_at',
]
if 'role' in alerts_df.columns:
    summary_cols.insert(2, 'role')
if 'department' in alerts_df.columns:
    summary_cols.insert(3, 'department')

summary_cols = [c for c in summary_cols if c in alerts_df.columns]

alerts_df[summary_cols].to_csv(ALERTS_CSV_PATH, index=False)

print(f'✅ alerts_summary.csv saved.')
print(f'   Path    : {ALERTS_CSV_PATH}')
print(f'   Shape   : {alerts_df[summary_cols].shape}')
print(f'   Size    : {os.path.getsize(ALERTS_CSV_PATH)/1e3:.1f} KB')

In [ ]:
# ── Final summary ─────────────────────────────────────────────
print('=' * 60)
print('  NOTEBOOK 03 — FINAL SUMMARY')
print('=' * 60)
print(f'  LLM endpoint      : {PCAI_ENDPOINT}')
print(f'  Model             : {PCAI_MODEL}')
print()
print(f'  Users processed   : {len(batch)}')
print(f'  Alerts generated  : {len(alerts)}')
print(f'    OK              : {ok_count}')
print(f'    Partial         : {partial_count}')
print(f'    Failed          : {failed_count}')
print()
print(f'  Alert quality:')
print(f'    FP filtered     : {len(fp_filtered)} '
      f'({len(fp_filtered)/max(len(alerts_valid),1)*100:.1f}%)')
print(f'    Actionable      : {len(true_alerts)} '
      f'({len(true_alerts)/max(len(alerts_valid),1)*100:.1f}%)')
print()
print(f'  Threat type breakdown:')
for threat, count in alerts_valid['threat_type'].value_counts().items():
    print(f'    {threat:<22}: {count}')
print()
print(f'  Outputs saved:')
print(f'  → {ALERTS_JSON_PATH}')
print(f'  → {ALERTS_CSV_PATH}')
print('=' * 60)
print('\n🎯 Notebook 03 Complete!')
print('   → Open 04_dashboard_and_validation.ipynb to build')
print('     the SOC dashboard and validate against ground truth.')

---
## ✅ Notebook 03 — Summary

### What We Built
| Step | Output |
|------|--------|
| PCAI LLM client | Retry-safe `call_llm()` with auth |
| Context builder | Structured evidence block per user |
| Chain-of-thought prompt | 5-step reasoning template |
| JSON response parser | Validation + normalisation of LLM output |
| Batch alert generation | LLM alerts for all flagged users |
| False positive filtering | LLM-assessed benign/high-FP alerts separated |
| `alerts.json` | Full structured alert store |
| `alerts_summary.csv` | Flat summary for dashboard |

### Key Design Decisions
- 🌡️ **Temperature = 0.1** — minimises hallucination, maximises JSON consistency
- 🔁 **Retry logic** — 3 attempts with backoff handles transient PCAI timeouts
- 🧠 **Chain-of-thought** — forces the LLM to consider benign hypotheses before flagging
- 🔍 **SHAP in context** — grounds the LLM in specific feature evidence, not just scores
- 🧹 **FP filter** — LLM-assessed high false positive risk alerts are separated, not deleted

### Next Step
**Notebook 04** — Build the interactive SOC dashboard with Plotly + Gradio,
then validate detection performance against the CERT r4.2 ground truth answers.